# Task 1

In [7]:
# Read data
import pandas as pd

# Load the dataset
df_data1 = pd.read_csv("amazon_review_texts.csv")

# Display the first 5 rows
print("First 5 rows of the dataset:")
print(df_data1.head())

# Distribution of the 'score' column
print("\nScore Distribution:")
print(df_data1['score'].value_counts())

# Distribution of the 'category' column
print("\nCategory Distribution:")
print(df_data1['category'].value_counts())


First 5 rows of the dataset:
          pid helpful  score  \
0  B000GAYQL8     0/0      5   
1  B000IBNPDA     0/0      5   
2  B000J2HA16     0/0      5   
3  B000BDIQPM     0/0      5   
4  B000GZTH9E     0/3      4   

                                                text category  
0  GREAT WATCH AND GREAT LOOK. BIG FACE AND 4 DIF...    watch  
1  Bought this as a Christmas gift, my boyfriend ...    watch  
2  I love this watch! Its sporty, without looking...    watch  
3  Works great,looks nice,dont have to worry abou...    watch  
4  I need to change the watch wrist and I havent ...    watch  

Score Distribution:
score
5    2070
4     773
1     595
3     303
2     259
Name: count, dtype: int64

Category Distribution:
category
watch          1000
software       1000
electronics    1000
automotive     1000
Name: count, dtype: int64


# Task 2

In [9]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk import FreqDist

# # Downloading NLTK data (for first-time only)
# nltk.download("stopwords")
# nltk.download("punkt")

# get a set of stopwords
stopwords = set(nltk.corpus.stopwords.words("english"))

# lowercase
def before_token(documents):
    # Convert to lowercase
    lower = map(str.lower, documents)
    # remove puntuations
    punctuationless = list(map(lambda x: " ".join(re.findall('\\b\\w\\w+\\b',x)), lower))
    # remove numbers
    return list(map(lambda x:re.sub('\\b[0-9]+\\b', '', x), punctuationless))
# Initialize stemmer
stemmer = nltk.stem.PorterStemmer()


# define a function that preprocess a single document and returns a list of tokens
def preprocess(doc):
    tokens = []
    for token in doc.split():
        if token not in stopwords:
            tokens.append(stemmer.stem(token))
    return tokens

# preprocess all documents
processed = list(map(preprocess, before_token(df_data1['text'])))
print(processed[0])                 

['great', 'watch', 'great', 'look', 'big', 'face', 'differ', 'mode', 'enough', 'sturdi', 'bright', 'indiglo', 'light', 'awesom', 'awesom', 'awesom', 'also', 'militari', 'guy', 'given', 'watch', 'battlefield', 'approv']


In [10]:
# initialize a container of token frequencies
fdist = nltk.FreqDist()
                 
# Frequency distribution of all tokens
fdist = nltk.FreqDist([token for doc in processed for token in doc])
# Print word frequency stats
print("Unique tokens:", fdist.B())
print("Total tokens:", fdist.N())
print("Tokens that occurred only once:", len(fdist.hapaxes()))

# Top 10 frequent tokens
print("\nTop 10 frequent tokens:")
fdist.tabulate(10)
                 


Unique tokens: 10973
Total tokens: 193927
Tokens that occurred only once: 4701

Top 10 frequent tokens:
  watch     use     one    work    time    like product   great     get   would 
   2553    2476    1795    1605    1420    1375    1336    1318    1309    1217 


# Task 3


From the top 10 frequent words obtained in Task 2:
**watch, use, one, work, time, like, product, great, get, would**

The following words might not be useful for clustering or classification:

**use, one, time, like, get, would**  
  These words are general purpose and likely appear across all product categories.so, they do not carry specific semantic information that can help distinguish between categories such as *electronics*, *software* or *automotive*. 
  For example:
  -"use" can refer to using any product like a watch, a charger or a software app.
  -"one" is a generic number or pronoun.
  -"time" could refer to the concept of time in watches or otherwise, but without context, it lacks discriminative power.
  -"get" and "would" are common auxiliary or action verbs found in all kinds of product reviews.

These words tend to add **noise rather than useful signals** for machine learning models and should be considered for removal or down-weighting using techniques like TF-IDF.

On the other hand, category-specific terms like **watch** or **product** might still offer some insight depending on their frequency across specific categories.


# Task 4

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

processed_doc = list(map(" ".join, processed))
# normalization is needed for clustering
vectorizer = TfidfVectorizer(norm = 'l2',max_df=0.8, stop_words='english')
X = vectorizer.fit_transform(processed_doc)


# Output number of samples and features
print("n_samples: %d, n_features: %d" % X.shape)
print("TF-IDF matrix type:", type(X))


n_samples: 4000, n_features: 10833
TF-IDF matrix type: <class 'scipy.sparse._csr.csr_matrix'>


# Task 5

In [14]:
from sklearn.cluster import KMeans

k=4
# Run KMeans on the TF-IDF matrix
km = KMeans(n_clusters=k, max_iter=100, random_state=54321)
km.fit(X)

# examine the representative words for each cluster
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

#Printing top 10 representative words per cluster
for i in range(k):
    print(f"\nCluster {i + 1}:")
    for ind in order_centroids[i, :10]:
        print(f" {terms[ind]}")



Cluster 1:
 use
 product
 work
 great
 good
 instal
 program
 like
 softwar
 time

Cluster 2:
 watch
 look
 band
 time
 great
 wear
 love
 like
 nice
 price

Cluster 3:
 bed
 air
 inflat
 comfort
 pump
 sleep
 mattress
 deflat
 airb
 easi

Cluster 4:
 batteri
 charg
 charger
 power
 adapt
 appl
 camera
 canon
 work
 origin


# Task 6

The top 10 words for each cluster provide meaningful insights and mostly align well with the four product categories in the dataset: **watch, automotive, electronic and software**.

-**Cluster 1** includes terms such as *use, product, work, install, softwar, program*, which are commonly associated with **software** reviews. These words indicate functional descriptions and installation experiences, which are typical in software product feedback.


-**Cluster 2** features words like *watch, band, look, time, wear, nice, price*, clearly pointing toward the **watch** category. These are strongly indicative of fashion and design oriented feedback that consumers typically leave for watches.


 **Cluster 3** includes *bed, air, inflat, mattress, deflat, pump*, which is less directly aligned with the four categories. However, it may correspond to **automotive** products, particularly inflatable car mattresses or travel gear. While slightly off-track, it still reflects a subset of automotive-related reviews.
 

-**Cluster 4** includes *battery, charger, power, adapter, camera, canon*, which clearly represent **electronics**. These terms are consistent with gadgets, devices and brand-specific components in the electronics domain.

In Overall **3 out of 4 clusters** align very well with the intended product categories.
The **third cluster** seems to describe a niche product group (possibly auto-related gear), but not in a generalizable way.
The clustering algorithm has generally done a good job of capturing thematic coherence within the groups, especially considering this is unsupervised learning without true labels.

# Task 7

In [17]:
import gensim
from gensim import corpora
from gensim.models import LdaModel
from gensim import matutils

#Create word2id and id2word mappings using the fitted vectorizer
word2id = dict((k, v) for k, v in vectorizer.vocabulary_.items())
id2word = dict((v, k) for k, v in vectorizer.vocabulary_.items())

#Create a Gensim Dictionary and Corpus
dictionary = corpora.Dictionary()
dictionary.id2token = id2word
dictionary.token2id = word2id

# Converting the TF-IDF sparse matrix to a Gensim corpus
corpus = matutils.Sparse2Corpus(X, documents_columns=False)

#Build the LDA model with 4 topics
lda_model = LdaModel(corpus=corpus, num_topics=4, id2word=id2word, passes=20, random_state=42)

#Printing the topics
print("LDA Topics:")
topics = lda_model.print_topics(num_words=10)
for topic_num, topic in topics:
    print(f"\nTopic {topic_num + 1}:")
    print(topic)


LDA Topics:

Topic 1:
0.007*"use" + 0.006*"work" + 0.006*"product" + 0.005*"great" + 0.004*"good" + 0.004*"batteri" + 0.004*"time" + 0.004*"like" + 0.004*"easi" + 0.003*"instal"

Topic 2:
0.027*"watch" + 0.007*"band" + 0.006*"look" + 0.005*"love" + 0.005*"wear" + 0.004*"wrist" + 0.003*"nice" + 0.003*"great" + 0.003*"face" + 0.003*"beauti"

Topic 3:
0.006*"printer" + 0.002*"cartridg" + 0.002*"print" + 0.001*"cannon" + 0.001*"grandson" + 0.001*"ink" + 0.001*"matress" + 0.001*"epson" + 0.001*"pic" + 0.001*"hp"

Topic 4:
0.003*"mop" + 0.001*"la" + 0.001*"el" + 0.001*"nissan" + 0.001*"en" + 0.001*"es" + 0.001*"gasket" + 0.001*"producto" + 0.001*"radioand" + 0.001*"que"


# Task 8

#Topic 1:
Words like *use, work, product, batteri, instal* indicate general usability and functionality-related terms. This topic likely captures feedback on **software or electronics**, but it is quite broad and overlaps multiple categories.

#Topic 2:
This topic clearly aligns with the **watch** category. High-weight words like *watch, band, look, wrist, wear, face* strongly relate to appearance and fashion—typical for watch reviews.

#Topic 3:
Contains technical terms like *printer, cartridg, ink, epson, hp*, which are highly indicative of the **electronics** category, especially printers and related accessories.

#Topic 4:
This topic is noisy and less coherent. It includes terms like *nissan, gasket, mop, producto, que*—mixing automotive parts, cleaning items, and non-English words. While *nissan* and *gasket* hint at **automotive**, the presence of unrelated or multilingual terms weakens the topic's clarity.

##### K-Means Clustering:
**Strengths**: Clear topic-word groupings, especially for watches, electronics and software. The top 10 words in each cluster were more coherent and relevant to known product categories.

**Weaknesses**: Still showed some mixing of themes, but generally more aligned with the original categories.

##### LDA Topic Modeling:
**Strengths**: Captured dominant terms like *watch* and *printer* effectively in certain topics.

**Weaknesses**: Produced at least one weak and noisy topic (Topic 4) and topics like Topic 1 were too general and less discriminative.

In Conclusion: **K-Means clustering** appears to be **more effective** in this example, It formed more distinct and semantically interpretable clusters that closely aligned with the four product categories. In contrast, **LDA**, though useful for uncovering latent topics, struggled with overlapping language and introduced noise, likely due to the short and informal nature of review texts.


# Task 9

In [20]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn import metrics

#Define the TF-IDF vectorizer (with additional filtering)
vectorizer = TfidfVectorizer(
    max_df=0.8,
    min_df=2,  # min_df=2 ensures we remove tokens that appear in only 1 document
    stop_words='english'
)

#Prepare text and labels
texts = df_data1['text']
labels = df_data1['score']

#Initialize cross-validator and storage
skf = StratifiedKFold(n_splits=5)
f1 = []

# Cross-validation loop
fold = 0
for train_index, test_index in skf.split(texts, labels):
    fold += 1
    print(f"\n--- Fold {fold} ---")
    # Split data
    train_x, test_x = texts.iloc[train_index], texts.iloc[test_index]
    train_y, test_y = labels.iloc[train_index], labels.iloc[test_index]
    # Vectorize
    X_train = vectorizer.fit_transform(train_x)
    X_test = vectorizer.transform(test_x)
    # To Print number of features
    if fold == 1:
        print("Number of TF-IDF features:", X_train.shape[1])
    # Train model
    clf = SGDClassifier(random_state=fold)
    clf.fit(X_train, train_y)
    # Predict
    pred_y = clf.predict(X_test)

    # classification results
    for line in metrics.classification_report(test_y, pred_y).split("\n"):
        print(line)
    f1.append(metrics.f1_score(test_y, pred_y, average='weighted'))
print("Average F1: %.2f" % np.mean(f1))


--- Fold 1 ---
Number of TF-IDF features: 7524
              precision    recall  f1-score   support

           1       0.74      0.38      0.50       119
           2       0.27      0.06      0.10        51
           3       0.14      0.05      0.07        61
           4       0.30      0.20      0.24       155
           5       0.63      0.91      0.74       414

    accuracy                           0.57       800
   macro avg       0.42      0.32      0.33       800
weighted avg       0.52      0.57      0.52       800


--- Fold 2 ---
              precision    recall  f1-score   support

           1       0.47      0.60      0.53       119
           2       0.17      0.08      0.11        52
           3       0.07      0.07      0.07        60
           4       0.30      0.25      0.28       155
           5       0.69      0.73      0.71       414

    accuracy                           0.53       800
   macro avg       0.34      0.35      0.34       800
weighted avg 

# Task 10

In [21]:

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import SGDClassifier
from sklearn import metrics
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

#Create new binary target variable
# 4 or 5 -> satisfied (1), 1 to 3 -> not satisfied (0)
df_data1['satisfaction'] = df_data1['score'].apply(lambda x: 1 if x >= 4 else 0)

# Use same TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_df=0.8,
    min_df=2,
    stop_words='english'
)

texts = df_data1['text']
labels = df_data1['satisfaction']

#Setup Stratified K-Fold
skf = StratifiedKFold(n_splits=5)
f1= []

fold = 0
for train_index, test_index in skf.split(texts, labels):
    fold += 1
    print(f"\n--- Fold {fold} ---")
    
    train_x, test_x = texts.iloc[train_index], texts.iloc[test_index]
    train_y, test_y = labels.iloc[train_index], labels.iloc[test_index]
    # Vectorize
    X_train = vectorizer.fit_transform(train_x)
    X_test = vectorizer.transform(test_x)
    # Print number of features in first fold
    if fold == 1:
        print("Number of TF-IDF features:", X_train.shape[1])

    # Training classifier
    clf = SGDClassifier(random_state=fold)
    clf.fit(X_train, train_y)
    
    # Predicting and evaluate
    pred_y = clf.predict(X_test)
    print(metrics.classification_report(test_y, pred_y))

    f1 = metrics.f1_score(test_y, pred_y, average='weighted')
    f1_scores.append(f1)

# Final average F1 score
print("\nAverage Weighted F1 Score across 5 folds: %.4f" % np.mean(f1_scores))


--- Fold 1 ---
Number of TF-IDF features: 7519
              precision    recall  f1-score   support

           0       0.84      0.23      0.35       231
           1       0.76      0.98      0.86       569

    accuracy                           0.76       800
   macro avg       0.80      0.60      0.61       800
weighted avg       0.78      0.76      0.71       800


--- Fold 2 ---
              precision    recall  f1-score   support

           0       0.64      0.70      0.67       231
           1       0.87      0.84      0.86       569

    accuracy                           0.80       800
   macro avg       0.76      0.77      0.76       800
weighted avg       0.80      0.80      0.80       800


--- Fold 3 ---
              precision    recall  f1-score   support

           0       0.65      0.69      0.67       231
           1       0.87      0.85      0.86       569

    accuracy                           0.80       800
   macro avg       0.76      0.77      0.76     

# Task 11

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn import metrics
import numpy as np

#Load opinion lexicon
lexicon = dict()

# Load negative words
with open("opinion-lexicon-English/negative-words.txt", "r") as f:
    for line in f:
        if not line.startswith(";") and line.strip():
            lexicon[line.strip()] = -1

# Load positive words
with open("opinion-lexicon-English/positive-words.txt", "r") as f:
    for line in f:
        if not line.startswith(";") and line.strip():
            lexicon[line.strip()] = 1


# To print the top 5 entries
for i, (k, v) in enumerate(lexicon.items()):
    print(k, v)
    if i > 4: break


2-faced -1
2-faces -1
abnormal -1
abolish -1
abominable -1
abominably -1


In [24]:
# Create vocabulary list for vectorizer
lexicon_vocab = list(lexicon.keys())

print("Sample entries from lexicon vocabulary:")
print(lexicon_vocab[:5])

Sample entries from lexicon vocabulary:
['2-faced', '2-faces', 'abnormal', 'abolish', 'abominable']


In [25]:
#Prepare text and labels
texts = df_data1['text']
labels = df_data1['satisfaction']  # from Task 10

#Define vectorizer with constrained vocabulary
vectorizer = TfidfVectorizer(
    max_df=0.8,
    min_df=2,
    stop_words='english',
    vocabulary=lexicon_vocab  # To Restrict vocabulary to lexicon words
)

#Set up cross-validation
skf = StratifiedKFold(n_splits=5)
f1_scores = []

fold = 0
for train_index, test_index in skf.split(texts, labels):
    fold += 1
    print(f"\n--- Fold {fold} ---")

    train_x, test_x = texts.iloc[train_index], texts.iloc[test_index]
    train_y, test_y = labels.iloc[train_index], labels.iloc[test_index]

    # Vectorize
    X_train = vectorizer.fit_transform(train_x)
    X_test = vectorizer.transform(test_x)

    if fold == 1:
        print("Number of features from lexicon vocabulary:", X_train.shape[1])

    # Train classifier
    clf = SGDClassifier(random_state=fold)
    clf.fit(X_train, train_y)

    # Predict and evaluate
    pred_y = clf.predict(X_test)
    print(metrics.classification_report(test_y, pred_y))

    f1 = metrics.f1_score(test_y, pred_y, average='weighted')
    f1_scores.append(f1)

#Average weighted F1 score
print("\nAverage Weighted F1 Score across 5 folds: %.4f" % np.mean(f1_scores))


--- Fold 1 ---
Number of features from lexicon vocabulary: 6786
              precision    recall  f1-score   support

           0       0.72      0.42      0.53       231
           1       0.80      0.93      0.86       569

    accuracy                           0.78       800
   macro avg       0.76      0.67      0.69       800
weighted avg       0.77      0.78      0.76       800


--- Fold 2 ---
              precision    recall  f1-score   support

           0       0.68      0.65      0.66       231
           1       0.86      0.88      0.87       569

    accuracy                           0.81       800
   macro avg       0.77      0.76      0.76       800
weighted avg       0.81      0.81      0.81       800


--- Fold 3 ---
              precision    recall  f1-score   support

           0       0.66      0.65      0.66       231
           1       0.86      0.86      0.86       569

    accuracy                           0.80       800
   macro avg       0.76      0.

# Task 12

In **Task 10**, the average weighted F1 score across 5 folds was **0.7782** using a standard TF-IDF vectorizer applied to all terms in the review text.

In **Task 11**, the average weighted F1 score improved slightly to **0.7903**, where the TF-IDF vectorizer was restricted to sentiment-bearing words from **Bing Liu’s opinion lexicon** (positive and negative words only).


**Yes**, the average F1 score **increased** from **0.7782** to **0.7903**.


The increase can be attributed to the **focused vocabulary** used in Task 11:

By limiting the TF-IDF vectorizer to **opinion-related words**, the model becomes more sensitive to sentiment cues that directly influence satisfaction.

Removing generic or irrelevant terms reduces noise, allowing the classifier to learn **more meaningful patterns** associated with satisfaction levels.

This shows that leveraging a **sentiment lexicon** can enhance predictive performance, especially in tasks that depend heavily on emotional tone or user sentiment, such as satisfaction prediction.


# Task 13 

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

#Using same TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_df=0.8,
    min_df=2,
    stop_words='english'
)

#Apply to TF-IDF and convert to dense matrix
X_tfidf_dense = vectorizer.fit_transform(df_data1['text']).todense()

# Standardize
X_scaled = StandardScaler().fit_transform(np.asarray(X_tfidf_dense))  # convert to array

#Apply PCA with whitening
pca_full = PCA(svd_solver='randomized', whiten=True).fit(X_scaled)

# Show how much variance each component explains
print("Explained variance ratio for first few components:")
print(pca_full.explained_variance_ratio_[:10])  # show first 10 for quick view

Explained variance ratio for first few components:
[0.00308208 0.00291359 0.00261601 0.00253572 0.00229716 0.00215668
 0.00213486 0.00195992 0.0018472  0.00174703]


# Task 14

**Principal Component Analysis (PCA)** is a dimensionality reduction technique used to simplify large datasets by transforming the original features into a new set of uncorrelated variables called **principal components**.

These components are ordered by the amount of variance they capture from the original data — meaning the first few components retain most of the meaningful patterns while reducing noise and redundancy.

PCA is useful for:

Reducing computational cost.

Improving model performance by eliminating irrelevant features.

Visualizing high-dimensional data in 2D or 3D space.

In our case, we use PCA to reduce the number of TF-IDF features before classification, while still retaining most


# Task 15

In [30]:
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import SGDClassifier
from sklearn import metrics
import numpy as np

#Run PCA once to determine number of components for 90% variance
pca_full = PCA(svd_solver='randomized', whiten=True)
pca_full.fit(X_scaled)

explained = pca_full.explained_variance_ratio_
sum_variance = 0.0
n_components = 0

for var in explained:
    sum_variance += var
    n_components += 1
    if sum_variance >= 0.90:
        break

print(f"\nNumber of principal components needed to explain at least 90% variance: {n_components}")


Number of principal components needed to explain at least 90% variance: 2239


In [32]:
# Run PCA again with identified number of components
pca_final = PCA(n_components=n_components, svd_solver='randomized', whiten=True)
X_pca = pca_final.fit_transform(X_scaled)

# Preparing labels
labels = df_data1['satisfaction']

# Perform 5-fold cross-validation on PCA-transformed data
skf = StratifiedKFold(n_splits=5)
f1_scores = []

fold = 0
for train_index, test_index in skf.split(X_pca, labels):
    fold += 1
    print(f"\n--- Fold {fold} ---")
    
    train_x, test_x = X_pca[train_index], X_pca[test_index]
    train_y, test_y = labels.iloc[train_index], labels.iloc[test_index]
    
    clf = SGDClassifier(random_state=fold)
    clf.fit(train_x, train_y)
    pred_y = clf.predict(test_x)

    print(metrics.classification_report(test_y, pred_y))
    
    f1 = metrics.f1_score(test_y, pred_y, average='weighted')
    f1_scores.append(f1)

#Final average F1 score
print("\nAverage Weighted F1 Score across 5 folds: %.4f" % np.mean(f1_scores))



--- Fold 1 ---
              precision    recall  f1-score   support

           0       0.87      0.06      0.11       231
           1       0.72      1.00      0.84       569

    accuracy                           0.73       800
   macro avg       0.79      0.53      0.47       800
weighted avg       0.76      0.72      0.63       800


--- Fold 2 ---
              precision    recall  f1-score   support

           0       0.54      0.09      0.15       231
           1       0.72      0.97      0.83       569

    accuracy                           0.71       800
   macro avg       0.63      0.53      0.49       800
weighted avg       0.67      0.71      0.63       800


--- Fold 3 ---
              precision    recall  f1-score   support

           0       0.70      0.12      0.21       231
           1       0.73      0.98      0.84       569

    accuracy                           0.73       800
   macro avg       0.72      0.55      0.52       800
weighted avg       0.72   

# Task 16

In **Task 10**, using the full TF-IDF feature set, the average weighted F1 score across 5 folds was **0.7782**.

In **Task 15**, after applying **PCA** to reduce dimensionality (retaining components that explained at least 90% of the variance), the average weighted F1 score **dropped** to **0.6436**.

here the F1 score **decreased** after applying PCA.

so, While PCA helps reduce dimensionality and remove noise, it can also distort interpretability and dilute feature relevance for classification tasks — especially when: the original TF-IDF features contain **sparse, meaningful token patterns** that PCA compresses into components not easily aligned with class boundaries. and the classifier (SGD) benefits more from the raw high-dimensional space where text features are distinctly represented.

here, In our case, applying PCA **removed important discriminative signals** needed to distinguish between satisfied and unsatisfied reviews, leading to a lower classification performance.